# 08 · Contamination checks

**Continuation test:** the model receives the first half of a test text; we record how much of the true second half it reproduces verbatim. **Perturbation test:** the most label-discriminative words (chi-square on the training set) are masked; markedly higher accuracy on masked **test** texts than on masked **validation** texts suggests memorisation.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import numpy as np, pandas as pd
from mad_indo.llm import setup_ollama, OllamaClient
from mad_indo import contamination as C, prompts
from mad_indo.extract import extract_label
lc, cc = cfg["llm"], cfg["contamination"]
setup_ollama(list(lc["models"].values()), host=lc["host"], num_parallel=lc["parallel_requests"])
client = OllamaClient(lc["host"], num_ctx=lc["num_ctx"], num_predict=64)

Installing Ollama ...
Starting Ollama server (parallel=4) ...
Pulling qwen2.5:7b ...
Pulling llama3.1:8b ...
Pulling gemma2:9b ...
Ollama ready | ollama version is 0.40.16 | models: qwen2.5:7b, llama3.1:8b, gemma2:9b


In [3]:
rows = []
for task in ["smsa", "emot"]:
    L = cfg["tasks"][task]["labels"]
    tr = data.load_split(task, "train")
    test = data.load_split(task, "test").sample(min(cc["n_samples"], 440), random_state=0)
    valid = data.load_split(task, "valid").sample(min(cc["n_samples"], 440), random_state=0)
    terms = C.discriminative_terms(tr.text, tr.label, k=cc["n_mask_terms"])
    for mk, tag in lc["models"].items():
        ov = []
        for t in test.text:
            pre, post = C.split_half(t)
            ov.append(C.overlap_ratio(post, client.chat(tag, C.continuation_prompt(pre))["content"]))
        acc = {}
        for name, df in [("test", test), ("valid", valid)]:
            preds = [extract_label(client.chat(tag, prompts.single_agent_messages("S1_zero_shot", 0, task, L, C.mask_text(t, terms)),
                     schema=prompts.json_schema(L))["content"], L)["label"] for t in df.text]
            acc[name] = np.mean(np.array(preds) == df.label.values)
        rows.append({"task": task, "model": mk, "mean_continuation_overlap": np.mean(ov),
                     "share_overlap_ge_0.8": np.mean(np.array(ov) >= 0.8),
                     "masked_acc_test": acc["test"], "masked_acc_valid": acc["valid"],
                     "gap_test_minus_valid": acc["test"] - acc["valid"]})
res = pd.DataFrame(rows).round(4); res.to_csv(RESULTS / "contamination.csv", index=False); res

,task,model,mean_continuation_overlap,share_overlap_ge_0.8,masked_acc_test,masked_acc_valid,gap_test_minus_valid
0,smsa,qwen,0.0100,0.0,0.705,0.560,0.145
1,smsa,llama,0.0124,0.0,0.685,0.625,0.060
2,smsa,gemma,0.0228,0.0,0.735,0.650,0.085
3,emot,qwen,0.0051,0.0,0.615,0.555,0.060
4,emot,llama,0.0050,0.0,0.590,0.510,0.080
5,emot,gemma,0.0106,0.0,0.655,0.610,0.045
